# Topical Lectures, Controls
## Real drone, part 2: flying with the drone's own controllers

Andreas Freise, 31.10.2026

### Goal

Before we build our own controller, we fly with the ones built into the drone. The firmware of the Crazyflie contains the complete cascade, position, velocity and attitude loops. From Python we only say where to go, for example, 'forward half a metre', 'turn', 'hold'. In this notebook you fly a few such paths, look at what the drone recorded, and get a feeling for what the on-board loops do well and where they are slow. The recorded data tell you how still the drone hovers and how well it returns to its start.

At the end of this notebook you have

* flown the drone along paths you wrote yourself, under the on-board controllers,
* seen how the firmware executes a move: the setpoint travels at constant speed and the drone follows it, and what happens at the end of a move,
* measured how still the hover is and how well the drone returns to its start,
* saved the flights as files, named after your drone and the time, as every flight of the afternoon.

### How the firmware flies

The drone measures its position with the flow deck: the optical flow of the floor and the height from the range sensor, combined with the accelerometer and the gyroscope in a Kalman filter. Its position controller turns the error to a setpoint into a velocity, its velocity loop into a tilt, its attitude loop into motor commands. `crazyflie.fly_path` sends it velocity setpoints at a fixed rate, in the frame of the drone: forward is where the motors M1 and M4 are. The two blue LEDs are at the back. The log records the position in the frame of the room, with the origin where the drone took off and `+x` where it pointed at that moment.

### Safety

* Fly only in the marked area. One person at the keyboard, one watching the drone, nobody within arm's length while the motors run.
* The person at the keyboard keeps the abort window in front, clicked once so that it takes the keys. **Enter** aborts and lands. The stop button of the cell also lands; press it once only. The window closes by itself after the landing.
* The flight lands by itself when the drone leaves a 1.5 m box around the start, when no state arrives any more, when the battery drops below 3.1 V under load, or when the flow deck stops delivering.
* After a tumble or crash: run the status cell; if it says locked, the cell after it clears the lock.
* Full battery. Below 3.5 V at rest the drone lacks thrust.

In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import crazyflie

%matplotlib inline

# The label of your drone as it reads on the sticker:
# LABEL = 2, or LABEL = 'A' for a letter.
LABEL = None
URI = crazyflie.drone_uri(LABEL)


## Status check

Switch the drone on, put it on the floor of the flying area and run the cell. It prints the battery voltage, whether the flow deck is detected, the flags of the supervisor, and whether the position estimate stays put on the ground.

* `flow deck: NOT attached`: the deck is not seated properly.
* `is locked: yes`: the supervisor has locked the motors after a tumble, a crash or a hard landing, which happens more often on a hard floor. Run the cell after this one, which clears the lock and shows the status again. If it stays locked, switch the drone off and on.
* `WARNING: the position estimate runs away` or `WARNING: the estimate does not follow the height sensor`: the flow deck has stopped delivering. Switch the drone off and on, connect again.
* `No drone answers at ...`: the drone is off, asleep or not on the channel associated with its label.


In [ ]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)

In [ ]:
# Only when the status says locked: clear the lock, then check again.
with crazyflie.connect(URI) as scf:
    crazyflie.recover(scf)
    crazyflie.status(scf)

## Test of the abort

Run this once per session. A dummy loop of 10 s opens the same abort window as the flights. Click into the window, press Enter, and the loop must report the abort. If it does not, do not fly from this program.


In [ ]:
crazyflie.bench_test_abort()

## The first flight: forward and back

A path is a list of moves: `('forward', 0.5)` is half a metre forward, `('left', 0.5)`, `('back', 0.5)` and `('right', 0.5)` likewise, `('up', 0.2)` and `('down', 0.2)` in metres, `('turn', 90)` turns 90 degrees to the left, `('hold', 2.0)` hovers for 2 s. `crazyflie.fly_path` takes off to 0.5 m, hovers, flies the moves at `speed` metres per second, holds and lands. The printout gives the elapsed time at each stage.

Put the drone flat on the floor with its forward side into the free space. The first path goes 0.5 m forward, holds for 2 s and comes back. The flight is saved under the label `firmware_fwd`; the files of every flight are named after your drone, the time and such a label.

The work by the cell ends with the landing. When the drone is down, the printout says `landed` and the abort window closes.


In [ ]:
PATH = [('forward', 0.5), ('hold', 2.0), ('back', 0.5)]

run = crazyflie.fly_path(PATH, uri=URI, speed=0.3)
name = crazyflie.save(run, 'firmware_fwd')


## What the drone recorded

Left: position against time, measured and planned. The planned line is the setpoint the firmware was given: it travels at constant speed and stops. Watch the measured line at the end of a move: does the drone stop where the setpoint stops, overshoot, or arrive late? Middle: the path in the plane. Right: the commanded velocity and the measured one, the response of the firmware's velocity loop, the loop whose response time you will measure in part 4.

The printout gives two numbers to keep: how still the hover was before the path, and how far from the planned position the drone ended.


In [ ]:
def show_path(name):
    """Position against time with the planned path, the path in the plane, and the velocities."""
    r = crazyflie.load_run(name)
    pid = r['pid']
    t = pid['t_s']
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={'width_ratios': [2, 1.2, 2]})
    for a in 'xy':
        line, = ax[0].plot(t, pid[a] - pid[f'target_{a}'][0], label=f'{a} measured')
        ax[0].plot(t, pid[f'target_{a}'] - pid[f'target_{a}'][0], ls='--', color=line.get_color(), lw=0.8, label=f'{a} planned')
    ax[0].set_xlabel('t [s]')
    ax[0].set_ylabel('position from start [m]')
    ax[0].legend(loc='upper left', fontsize=8)
    ax[0].grid()
    ax[1].plot(pid['x'] - pid['x'][0], pid['y'] - pid['y'][0], label='measured')
    ax[1].plot(pid['target_x'] - pid['target_x'][0], pid['target_y'] - pid['target_y'][0], 'k--', lw=0.8, label='planned')
    ax[1].plot(0, 0, 'k+', ms=12, mew=2)
    ax[1].set_xlabel('x [m]')
    ax[1].set_ylabel('y [m]')
    ax[1].set_aspect('equal')
    ax[1].legend(fontsize=8)
    ax[1].grid()
    for a in 'xy':
        line, = ax[2].plot(t, pid[f'fb_{a}'], label=f'{a} setpoint')
        ax[2].plot(t, pid[f'v{a}'], color=line.get_color(), alpha=0.5, label=f'{a} measured')
    ax[2].set_xlabel('t [s]')
    ax[2].set_ylabel('velocity [m/s]')
    ax[2].legend(loc='upper left', fontsize=8)
    ax[2].grid()
    plt.tight_layout()
    hold = pid['step'] < 0.5
    print(f"hover before the path: x std {1000 * np.std(pid['x'][hold]):.1f} mm, y std {1000 * np.std(pid['y'][hold]):.1f} mm; "
          f"end of the path: {100 * np.hypot(pid['x'][-1] - pid['target_x'][-1], pid['y'][-1] - pid['target_y'][-1]):.1f} cm from the planned position")

show_path(name)


## Play

Write your own paths, fly them, look at the recording, and save each under a label that says what it was. Change one thing at a time. Some things worth trying:

* **A square.** `[('forward', 0.5), ('left', 0.5), ('back', 0.5), ('right', 0.5)]`: four moves, no turn, the drone keeps pointing forward. Does it come back to the start? The printout says how far off it ended.
* **The same square with turns.** `[('forward', 0.5), ('turn', 90)] * 4`: now the drone turns left at each corner and flies forward each time. The moves are in the frame of the drone, the recording is in the frame of the room. Compare the two paths in the plane.
* **Speed.** The same path at `speed=0.15` and at `speed=0.5`. What changes at the corners, and how long after the last move does the drone stand still?
* **A hold.** `[('hold', 8.0)]`: eight seconds of hover and nothing else. The spread of the position during the hold is what the on-board loop achieves with this sensor.

The run label goes into the file names together with the label of your drone and the time, `flight_A_1532_square_pid.csv`, so that every flight of the afternoon can be put side by side later.


In [ ]:
PATH = [('forward', 0.5), ('left', 0.5), ('back', 0.5), ('right', 0.5)]
RUN_LABEL = 'square'

run = crazyflie.fly_path(PATH, uri=URI, speed=0.3)
name = crazyflie.save(run, RUN_LABEL)
show_path(name)